# Extraction of watersheds
This function:

- Rasterises polygons (GeoDataFrame) in a given 0.01° grid (already saved)

- Extracts the ID of the polygon that intersects each cell

- Returns nothing if no entity crosses a cell (no 0, just empty/NaN)

- Produces a latitude × longitude matrix containing int64

In [1]:
# importing all libraries needed for the extraction
import zipfile
import geopandas as gpd
import os
import re
import pandas as pd
from tqdm import tqdm

In [ ]:
# connection to the drive where the rasters are kept and where the results will be saved
from google.colab import drive
drive.mount('/content/drive')

In [20]:
# Here the parameters for the code to run smoothly are defined
# this code will loop on all the grids saved in a zipped folder (less place used and still efficient)

# === PARAMETERS ===
# path to the polygons' folder
POLY_FOLDER = "/content/drive/watershed"
# Name of the Id field, the one that we extract
ID_FIELD = "HYBAS_ID"
# path to the folder where you want the results to be saved as .csv in your drive
OUTPUT_FOLDER = "/content/drive/"
# Path to the zip file containing the grids of the different zones
ZIP_PATH = "/content/drive/grid_01_zone11.zip"

# if the output folder does not exist, it is created
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [12]:
# This function loop on all the watershed polygons and register all the grids one polygon intersects and the intersection's area. At the end, the one thats takes the biggest area is saved based on lat and long.
def indexation_polygones_max_area(grille_zone_path, shapefile_path, champ_id):
    # Load data
    grille = gpd.read_file(grille_zone_path)
    polygones = gpd.read_file(shapefile_path)

    # Get the grid's spatial index
    sindex = grille.sindex

    # Create dictionary to get all results {idx_cellule: (id_polygone, area_intersection)}
    cell_dict = {}

    # Following progress
    tqdm.pandas(desc="Polygons' processing")

    # loop on the watershed's polygons
    for poly in tqdm(polygones.itertuples(index=False), total=len(polygones), desc="Polygons"):
        # get the polygon's geometry
        geom = poly.geometry
        # get the id
        val = getattr(poly, champ_id)

        # get the intersection between the polygon and the cells
        possible_matches_index = list(sindex.intersection(geom.bounds))
        intersecting = grille.iloc[possible_matches_index]
        intersecting = intersecting[intersecting.intersects(geom)]

        # save the intersection with the biggest area for each cell
        for idx, cell_geom in intersecting.geometry.items():
            intersection = cell_geom.intersection(geom)
            area = intersection.area
            if area > 0:  # Ignorer si aucune surface réelle
                if idx not in cell_dict or area > cell_dict[idx][1]:
                    cell_dict[idx] = (val, area)

    # Results saved as dataframe
    grille["watershed_id"] = grille.index.map(lambda idx: cell_dict[idx][0] if idx in cell_dict else None)
    grille["latitude"] = round(grille.centroid.y, 3)
    grille["longitude"] = round(grille.centroid.x, 3)

    df = grille[["latitude", "longitude", "watershed_id"]].copy()

    # This takes care of the possible duplicates of lat and long
    df = df.groupby(["latitude", "longitude"], as_index=False).agg({
        "watershed_id": "first"
    })
    # return the results dataframe
    return df

In [ ]:
# Search the zip file with the grids
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    # Only choose the files in format .gpkg
    gpkg_files = [f for f in archive.namelist() if f.endswith(".gpkg")]

    # Loop on all the grids
    for filename in gpkg_files:

        # Extract the zone's number from the name of the file
        match = re.search(r"grid_001_zone(\d+)", filename) # rename the search depending on the filename format
        if match:
            zone_id = int(match.group(1))
        else:
            print(f"❌ Impossible to find the zone's number in this file : {filename}")
            continue

        print(f"\n➡️ Extraction zone_id {zone_id}")

        # Read the 0.01 grid and the watershed shape of the same zone
        GRILLE_PATH = f"/vsizip/{ZIP_PATH}/{filename}"
        POLY_PATH = os.path.join(POLY_FOLDER, f"watershed_zone_{zone_id}.shp")

        # Get all the ids
        try:
            df_zone = indexation_polygones_max_area(GRILLE_PATH, POLY_PATH, ID_FIELD)
        except Exception as e:
            print(f"❌ Error processing for {zone_id} : {e}")
            continue

        # Save as .csv
        try:
            # create a double entry matrix based on lat and long
            pivoted = df_zone.pivot(index="latitude", columns="longitude", values="watershed_id")
            # keep the lat and long in the right order (N-S and W-E)
            pivoted = pivoted.sort_index(ascending=False)
            # save as csv
            pivoted_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{ID_FIELD}_zone_{zone_id}_pivoted.csv")
            pivoted.to_csv(pivoted_csv)
            print(f"✅ Double entry table saved at :  {pivoted_csv}")
        except Exception as e:
            # in case there was a problem to create the double entry table, the dataframe will be saved as is
            output_csv = os.path.join(OUTPUT_FOLDER, f"Extraction_{ID_FIELD}_zone_{zone_id}.csv")
            df_zone.to_csv(output_csv, index=False)
            print(f"❌ Double entry was not created : {e}\n✅ Data still saved in brut at {output_csv}.")